In [ ]:
import pandas as pd
import numpy as np

rentals_raw = pd.read_csv('data/bike_rentals.csv')
rentals = rentals_raw.copy()

print(rentals.shape)
print(rentals.dtypes)

## Lab 5 - Part 1
### Ex 1

In [ ]:
print(rentals.columns[rentals.isna().all()])

rentals = rentals.dropna(axis=1, how='all')

print(len(rentals.columns[rentals.isna().all()]))
print(rentals.shape)
print(rentals.dtypes)

#### initial shape: (403, 15). price has type str.
#### 3 columns contain only empty values: ['coupon_code', 'damage_notes', 'late_fee_note']
#### shape after removal: (403, 12). 0 empty columns.
### Ex 2

In [ ]:
print(rentals['price'].head())

rentals['price'] = (rentals['price']
                    .str.strip()
                    .str.replace('SEK', '')
                    .str.replace(',', '')
                    .astype(float))

print(rentals['price'].head())
print(rentals['price'].describe())
rentals[['rental_id','bike_type','status','price']].sort_values('price',ascending=False).head(3)

#### describe(): count 352, mean 847.3, std 633.4, min 154.5, median 668.2, max 3401.0
#### Three most expensive: R-5282 (Cargo bike, 3401.0), R-5209 (Cargo bike, 2979.5), R-5378 (E-bike, 2941.0)
### Ex 3

In [ ]:
print(rentals['price'].isna().sum())
print((rentals['price'].isna().sum() / rentals['price'].count()) *100, '%')

pd.crosstab(rentals['price'].notna(), rentals['status'])

#### 51 rentals missing price, 14.49%.
#### We see that most of the missing prices are concentrated to status: Cancelled. 47/51. So these are correctly showing no payment.
#### 4/51 under status: Complete. These are probably faulty values.
#### We leave them all as missing for now. With the faulty values it is uncertain what part is incorrect.
### Ex 4

In [ ]:
t_f_col = [c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {'t', 'f'}]
print(t_f_col)

for c in t_f_col:
    rentals[c] = rentals[c].map({'t': True, 'f': False})
    print(c,
        'missing', rentals[c].isna().sum(),
        'False', rentals[c].isin([False]).sum())
    
rentals.groupby('is_member').agg(n=('price','count'),median_p=('price','median'))

#### columns with t,f values: ['is_member', 'insured', 'paid_online']. status has 3 different values and not bool-like.
#### is_member missing 0 False 288, insured missing 3 False 181, paid_online missing 0 False 111
#### is_member, rentals, median: False	248	756.25 , True	104	558.00
### Ex 5

In [ ]:
date_cols = ['start_date','end_date']

for c in date_cols:
    print(c, 'before', rentals[c].isna().sum())
    rentals[c] = pd.to_datetime(rentals[c], errors='coerce')
    print(c, 'after', rentals[c].isna().sum())

rentals['rental_days'] = (rentals['end_date'] - rentals['start_date']).dt.days

print(rentals.loc[(rentals['rental_days'] <  0),['rental_id','rental_days']])
rentals['rental_days'].describe().round(1)

#### missing dates: before both 0. after star_date=2 end_date=0. invalid dates.
#### describe: count    401.0 mean       3.0 min       -2.0 max        7.0
#### 2 rentals have a negative rental period delta, 6      R-5077         -1.0 196    R-5217         -2.0
### Ex 6

In [ ]:
print(rentals['rating'].isna().sum())
print(pd.crosstab(rentals['rating'].notna(), rentals['status']))

rentals['has_rating'] = rentals['rating'].notna()

print(rentals['has_rating'].value_counts())
print('median', rentals['rating'].median(),
      '\nmean', rentals['rating'].mean())

#### 115 missing. cancelled: 47, completed: 46, no-show: 22. completed without rating is where none was given.
#### has_rating: True 288, False 115. rating median: 5.0. mean: 4.23.
### Ex 7

In [ ]:
print(rentals['helmets_text'].value_counts(dropna=False))

rentals['child_helmet'] = rentals['helmets_text'].str.contains('child')
rentals['helmets_count'] = rentals['helmets_text'].str.extract(r'(\d+)', expand=False)

print(rentals['child_helmet'].value_counts(),
      rentals['helmets_count'].value_counts(dropna=False))

rentals['helmets_count'] = rentals['helmets_count'].where(~rentals['helmets_text'].str.contains('No helmets'), 0)

print(rentals['helmets_count'].value_counts(dropna=False))

#### 1 helmet 146, 2 helmets 127, No helmets 59, 1 child helmet 47, 3 helmets 20, missing 4.
#### With where(~text.str.contains('No helmets'), 0) it becomes 0. helmets_count: 0 → 59, 1 → 193, 2 → 127, 3 → 20, missing 4
#### child_helmet True: 47.
### Ex 8

In [ ]:
for c in ['station','bike_type','status']:
    print(c, rentals[c].value_counts(dropna=False).count())
    print(c, rentals[c].str.strip().str.lower().value_counts().count())

rentals['station'] = rentals['station'].str.strip().str.lower().str.title()
print(rentals['station'].value_counts(dropna=False).count())

print(rentals.duplicated().sum())
print(rentals.loc[rentals.duplicated(keep=False),['rental_id','bike_type','rating']].sort_values('rental_id'))

rentals = rentals.drop_duplicates().reset_index(drop=True)

print(rentals.duplicated().sum())

rentals.to_csv('data/bike_rentals_clean.csv', index=False)
rentals.shape

#### station: 20 | 5. bike_type: 3 | 3. status: 3 | 3. Station could do with a bit 'o fixin'.
#### 3 duplicated rows. true duplicated. not in series but all values shared including id. Removed.
#### Final shape: (400, 16)